# r9p step C: train r9p from r8p
About 3-6 h.

In [ ]:
import os, glob, json, hashlib, shutil, subprocess
W = '/kaggle/working'
PUB, PRIV = f'{W}/public', f'{W}/private'   # public: counts only, safe to commit; private: real question text, never commit
os.makedirs(PUB, exist_ok=True)
os.makedirs(PRIV, exist_ok=True)
REPO, X
R8P_SHA = '38c289d351d6517b5cde1bc9fb64ea97bd33eee193639d6351edda52fdfd7a52'


def sh(cmd):
    """Runs one shell command; stops the notebook if it fails."""
    print('$', cmd, flush=True)
    if subprocess.run(cmd, shell=True).returncode:
        raise SystemExit(f'STOP: command failed: {cmd}')


def stop(msg):
    raise SystemExit('STOP: ' + msg)


def find(pattern):
    return sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))


def one(pattern, what):
    got = find(pattern)
    if not got:
        stop(f'attach {what} (no {pattern} under /kaggle/input)')
    return got[0]


def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


os.makedirs('/kaggle/temp', exist_ok=True)
if not os.path.exists(REPO):
    sh(f'git clone --depth 1 -b {BRANCH} https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}')
os.chdir(REPO)
tgz = find('real_questions_full.tgz')
if not tgz:
    stop('attach the private dataset kinthread-real-questions (it holds real_questions_full.tgz)')
sh(f'rm -rf data/real_questions && tar -xzf {tgz[0]}')
if not glob.glob('data/real_questions/*.jsonl'):
    stop('no real question files after unpacking real_questions_full.tgz')
print('repo and real questions ready')


def r8p_checkpoint():
    cands = find('best.pt')
    try:
        import kagglehub
        for h in ('anirudharavalli/notebook99ae0586bd/versions/7', 'anirudharavalli/notebook99ae0586bd/versions/5'):
            try:
                cands += glob.glob(f'{kagglehub.notebook_output_download(h)}/**/best.pt', recursive=True)
            except Exception as e:
                print(h, e)
    except Exception as e:
        print('kagglehub:', e)
    for p in cands:
        if sha(p) == R8P_SHA:
            return p
    stop(f'no best.pt with SHA-256 {R8P_SHA} among {cands}')


os.makedirs('runs/v10-r8p-final', exist_ok=True)
if not os.path.exists('runs/v10-r8p-final/best.pt'):
    shutil.copy(r8p_checkpoint(), 'runs/v10-r8p-final/best.pt')
print('r8p checkpoint ready', sha('runs/v10-r8p-final/best.pt')[:12])

sh('pip -q install -U gdown')
S = '/kaggle/temp/share'
os.makedirs(S, exist_ok=True)
if not os.path.isdir(f'{S}/dataset_v7'):
    sh(f'gdown -q --folder https://drive.google.com/drive/folders/1MNiQuramSX5ImytxEdvNEIN7SgFtT6k8 -O {S}/dataset_v7')
if not os.path.exists(f'{S}/star_r3.jsonl.gz'):
    sh(f'wget -q -O {S}/star_r3.jsonl.gz "https://drive.usercontent.google.com/download?id=1_qk0JL_lkzKBdv2vwHxWijI4yNrNPRtr&export=download&confirm=t"')
DATA, OUT = f'{S}/dataset_v7', f'{W}/runs/v10-r9p'
os.makedirs(OUT, exist_ok=True)
sh('cd data && sha256sum -c SHA256SUMS --ignore-missing --quiet')

In [ ]:
EXTRA = [f'{DATA}/reworded.jsonl.gz*2', 'data/mix_r8.jsonl.gz', f'{S}/star_r3.jsonl.gz*2',
         'data/balanced_mix_v1.jsonl.gz*2', 'data/detour_v1.jsonl.gz']
missing = [e for e in EXTRA if not os.path.exists(e.split('*')[0])]
if missing:
    stop(f'training files missing: {missing}')
sh('PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train_reader.py --arch v10 --run v10-r9p '
   f'--out {OUT} --pool {DATA} --init runs/v10-r8p-final/best.pt --prefix_lm --of_form --style_p 0.8 '
   '--steps 15000 --batch 64 --lr 3e-4 --warmup 500 --optim muon --amp --workers 2 --resume '
   '--eval_every 1500 --save_every 1000 --reworded ' + ','.join(EXTRA) + ' --reworded_dev data/reworded_dev.jsonl.gz '
   '--p_reworded 0.75 --ramp_steps 2000 --block 512 --real_dev 575 --select_real')

In [ ]:
sh(f'rm -f {OUT}/latest.pt')
best = f'{OUT}/best.pt'
if not os.path.exists(best):
    stop('training wrote no best.pt')
sh(f'python -m eval.run_real --ckpt {best} --split dev --adaptive 1.0 --hybrid --out {PRIV}/r9p_dev_quick.json')
quick = json.load(open(f'{PRIV}/r9p_dev_quick.json'))['score']['splits']
summary = {'step': 'C', 'r9p_sha256': sha(best), 'r9p_dev_quick_adaptive_hybrid': quick,
           'r8p_dev_same_setting': 'see colab/r8p_kaggle.ipynb output (dev, --adaptive 1.0 --hybrid)'}
json.dump(summary, open(f'{PUB}/C_summary.json', 'w'), indent=1)
print(json.dumps(summary, indent=1))
print('C DONE')